# Multi-Seed Ensemble — PubMedBERT

**Goal:** Train PubMedBERT 5 times with different random seeds, ensemble predictions
→ expected accuracy ~89.5%+ (up from 88.36%)

**How:**
1. Train Model 1 (seed=42), Model 2 (seed=123), Model 3 (seed=456), Model 4 (seed=789), Model 5 (seed=999)
2. All trained on same 4500 cleaned rows
3. Each predicts on test set
4. Average their probabilities (soft voting)
5. Final = argmax of averaged probabilities

**Time:** ~60-75 minutes (5 models trained)

**Setup:** Runtime → T4 GPU → Save

## Step 1: Install

In [ ]:
!pip install -q transformers torch openpyxl pandas scikit-learn

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Step 2: Upload `annotation_cleaned.xlsx`

In [ ]:
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

## Step 3: Load Data + Setup

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, classification_report, confusion_matrix,
                              matthews_corrcoef)
from collections import Counter
import random
import warnings
warnings.filterwarnings('ignore')

df = pd.read_excel('annotation_cleaned.xlsx')
df = df.dropna(subset=['Info (English)', 'Association Nature'])
df = df[df['Association Nature'].str.strip() != '']

df['text'] = ('[SPICE] ' + df['Spice'].astype(str).str.strip() + 
              ' [DISEASE] ' + df['Disease'].astype(str).str.strip() + 
              ' [INFO] ' + df['Info (English)'].astype(str).str.strip())
df['label'] = df['Association Nature'].astype(str).str.strip().str.lower()

X = df['text'].values
y = df['label'].values

# Single train/test split (80/20) to demonstrate ensemble cleanly
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Train dist: {dict(Counter(y_train))}')
print(f'Test dist: {dict(Counter(y_test))}')

MODEL_NAME  = 'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract'
MAX_LEN     = 256
BATCH_SIZE  = 16
EPOCHS      = 5
LR          = 2e-5
LABEL2ID    = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LABEL    = {0: 'negative', 1: 'neutral', 2: 'positive'}
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SEEDS = [42, 123, 456, 789, 999]  # 5 different random seeds

class SpiceDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = [LABEL2ID[l] for l in labels]
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_len,
                              padding='max_length', truncation=True, return_tensors='pt')
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long),
        }

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'\nReady. Will train {len(SEEDS)} models with seeds: {SEEDS}')

## Step 4: Train 5 Models with Different Seeds

In [ ]:
def train_and_predict(seed, X_tr, y_tr, X_te, y_te):
    """Train one PubMedBERT model with the given seed, return probabilities on test set."""
    set_seed(seed)
    print(f'  Training Model with seed={seed}...')

    train_ds = SpiceDataset(X_tr, y_tr, tokenizer, MAX_LEN)
    test_ds  = SpiceDataset(X_te, y_te, tokenizer, MAX_LEN)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID,
    ).to(DEVICE)

    counts = Counter(y_tr)
    total = len(y_tr)
    weights = torch.tensor(
        [total/(3*counts['negative']),
         total/(3*counts['neutral']),
         total/(3*counts['positive'])],
        dtype=torch.float
    ).to(DEVICE)
    loss_fn = torch.nn.CrossEntropyLoss(weight=weights)

    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, total_steps // 10, total_steps)

    # Train
    for epoch in range(1, EPOCHS+1):
        model.train()
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            outputs = model(input_ids=batch['input_ids'].to(DEVICE),
                            attention_mask=batch['attention_mask'].to(DEVICE))
            loss = loss_fn(outputs.logits, batch['label'].to(DEVICE))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()
        print(f'    Epoch {epoch}/{EPOCHS}: loss={total_loss/len(train_loader):.4f}')

    # Predict
    model.eval()
    all_probs = []
    with torch.no_grad():
        for batch in test_loader:
            outputs = model(input_ids=batch['input_ids'].to(DEVICE),
                            attention_mask=batch['attention_mask'].to(DEVICE))
            probs = torch.softmax(outputs.logits, dim=1)
            all_probs.append(probs.cpu().numpy())

    probs = np.concatenate(all_probs)
    preds = np.argmax(probs, axis=1)
    pred_labels = [ID2LABEL[p] for p in preds]
    acc = accuracy_score(y_te, pred_labels) * 100
    f1 = f1_score(y_te, pred_labels, average='weighted', zero_division=0) * 100

    del model
    torch.cuda.empty_cache()
    return probs, acc, f1

# Train all 5 models
all_probs = []
individual_results = []

print('=' * 65)
print('  Training 5 Models with Different Seeds')
print('=' * 65)

for i, seed in enumerate(SEEDS, 1):
    print(f'\n--- Model {i}/{len(SEEDS)} (seed={seed}) ---')
    probs, acc, f1 = train_and_predict(seed, X_train, y_train, X_test, y_test)
    all_probs.append(probs)
    individual_results.append({'seed': seed, 'accuracy': acc, 'f1': f1})
    print(f'  ✓ Model {i} done: Acc={acc:.2f}%  F1={f1:.2f}%')

print('\n✓ All 5 models trained')

## Step 5: Ensemble (Soft Voting)

In [ ]:
# Average probabilities across all 5 models
ensemble_probs = np.mean(all_probs, axis=0)
ensemble_preds = np.argmax(ensemble_probs, axis=1)
ensemble_labels = [ID2LABEL[p] for p in ensemble_preds]

# Metrics for ensemble
ens_acc = accuracy_score(y_test, ensemble_labels) * 100
ens_prec = precision_score(y_test, ensemble_labels, average='weighted', zero_division=0) * 100
ens_rec = recall_score(y_test, ensemble_labels, average='weighted', zero_division=0) * 100
ens_f1 = f1_score(y_test, ensemble_labels, average='weighted', zero_division=0) * 100

# Compute MCC
y_test_int = [LABEL2ID[l] for l in y_test]
ens_mcc = matthews_corrcoef(y_test_int, ensemble_preds)

print('=' * 65)
print('  RESULTS — Individual Models vs Ensemble')
print('=' * 65)
print(f"  {'Model':<20} {'Accuracy':>10} {'F1 Score':>10}")
print(f"  {'-'*20} {'-'*10} {'-'*10}")
for r in individual_results:
    print(f"  Seed {r['seed']:<14} {r['accuracy']:>9.2f}%  {r['f1']:>9.2f}%")
print(f"  {'-'*20} {'-'*10} {'-'*10}")
print(f"  {'ENSEMBLE (avg)':<20} {ens_acc:>9.2f}%  {ens_f1:>9.2f}%")
print()
print(f"  Individual avg: Acc={np.mean([r['accuracy'] for r in individual_results]):.2f}%  F1={np.mean([r['f1'] for r in individual_results]):.2f}%")
print(f"  Ensemble gain over avg individual: +{ens_acc - np.mean([r['accuracy'] for r in individual_results]):.2f}%")

print(f'\n  Per-Class Report:')
print(classification_report(y_test, ensemble_labels, digits=3, zero_division=0))

cm = confusion_matrix(y_test, ensemble_labels, labels=['positive','negative','neutral'])
print(f'  Confusion Matrix:')
print(f'               Pred-pos  Pred-neg  Pred-neu')
for i, label in enumerate(['positive','negative','neutral']):
    print(f'  True-{label:<10} {cm[i][0]:>8}  {cm[i][1]:>8}  {cm[i][2]:>8}')

## Step 6: Save Results

In [ ]:
with open('ensemble_results.txt', 'w') as f:
    f.write('=' * 65 + '\n')
    f.write('  MULTI-SEED ENSEMBLE RESULTS\n')
    f.write('=' * 65 + '\n\n')
    f.write('Individual Models:\n')
    for r in individual_results:
        f.write(f"  Seed {r['seed']:<5} Acc={r['accuracy']:.2f}%  F1={r['f1']:.2f}%\n")
    f.write(f'\nIndividual Average: Acc={np.mean([r["accuracy"] for r in individual_results]):.2f}%  F1={np.mean([r["f1"] for r in individual_results]):.2f}%\n')
    f.write(f'\nENSEMBLE (5 models averaged):\n')
    f.write(f'  Accuracy:    {ens_acc:.2f}%\n')
    f.write(f'  Precision:   {ens_prec:.2f}%\n')
    f.write(f'  Recall:      {ens_rec:.2f}%\n')
    f.write(f'  F1 Score:    {ens_f1:.2f}%\n')
    f.write(f'  MCC:         {ens_mcc:.3f}\n')
    f.write(f'\nGain from ensembling: +{ens_acc - np.mean([r["accuracy"] for r in individual_results]):.2f}%\n')
    f.write('=' * 65 + '\n')

print('✓ Saved ensemble_results.txt')

from google.colab import files
files.download('ensemble_results.txt')